In [ ]:
# pip install requests beautifulsoup4 pandas openpyxl undetected-chromedriver

import os
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
    NoSuchElementException,
)

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
OUTPUT_ROOT = r"OUTPUT"
OUTPUT_CSV_DIR = os.path.join(OUTPUT_ROOT, "CSV")
OUTPUT_EXCEL_DIR = os.path.join(OUTPUT_ROOT, "EXCEL")

REQUIRED_COLUMNS = [
    "Nomor Perkara",
    "Tanggal Register",
    "Klasifikasi Perkara",
    "Para Pihak",
    "Status Perkara",
    "Lama Proses",
]

REQUEST_TIMEOUT = 20
POLITE_DELAY = 0.8
CHROME_MAJOR_VERSION = 150  # update this if/when your Chrome auto-updates


# ---------------------------------------------------------------------------
# 1. HTTP SESSION (used only for the initial detection request)
# ---------------------------------------------------------------------------
def create_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
        )
    })
    return session


# ---------------------------------------------------------------------------
# 2. TEXT NORMALIZATION
# ---------------------------------------------------------------------------
def normalize_text(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()


# ---------------------------------------------------------------------------
# 3. COURT NAME DETECTION
# ---------------------------------------------------------------------------
def detect_court_name(soup):
    text = soup.get_text(" ", strip=True)
    match = re.search(r"PENGADILAN NEGERI [A-Z.\- ]+?(?=\s{2,}|\s+\d|Pembaharuan|$)", text)
    if match:
        return normalize_text(match.group(0))
    if soup.title and soup.title.string:
        t = normalize_text(soup.title.string)
        if "PENGADILAN" in t.upper():
            return t
    return "PENGADILAN NEGERI TIDAK DIKETAHUI"


# ---------------------------------------------------------------------------
# 4. PHI PAGE DETECTION
# ---------------------------------------------------------------------------
def looks_like_phi_table(soup):
    return detect_case_table(soup) is not None


def detect_phi_url(soup, base_url):
    for anchor in soup.find_all("a", href=True):
        text = normalize_text(anchor.get_text(" ", strip=True)).upper()
        if "HUBUNGAN INDUSTRIAL" in text or text == "PHI" or "(PHI)" in text:
            return urljoin(base_url, anchor["href"])

    heading = ""
    h_tag = soup.find(["h1", "h2", "h3"])
    if h_tag:
        heading = normalize_text(h_tag.get_text(" ", strip=True)).upper()
    elif soup.title and soup.title.string:
        heading = normalize_text(soup.title.string).upper()

    if "HUBUNGAN INDUSTRIAL" in heading and looks_like_phi_table(soup):
        return base_url

    raise ValueError(
        "Tidak dapat menemukan halaman PHI (Pengadilan Hubungan Industrial) "
        "dari URL yang diberikan."
    )


# ---------------------------------------------------------------------------
# 5. TABLE DETECTION + HEADER MAPPING
# ---------------------------------------------------------------------------
def detect_case_table(soup):
    for table in soup.find_all("table"):
        header_text = normalize_text(table.get_text(" ", strip=True)).upper()
        hits = sum(1 for col in REQUIRED_COLUMNS if col.upper() in header_text)
        if hits >= 4:
            return table
    return None


def get_header_mapping(table):
    header_row = table.find("tr")
    mapping = {}
    if header_row is None:
        return mapping
    cells = header_row.find_all(["th", "td"])
    for idx, cell in enumerate(cells):
        header_text = normalize_text(cell.get_text(" ", strip=True)).upper()
        if header_text:
            mapping[header_text] = idx
    return mapping


def find_column_index(mapping, name):
    name_upper = name.upper()
    if name_upper in mapping:
        return mapping[name_upper]
    for key, idx in mapping.items():
        if name_upper in key or key in name_upper:
            return idx
    return None


# ---------------------------------------------------------------------------
# 6. PARTY EXTRACTION (fixed: no separator in get_text, avoids splitting on
#    nested tags like <b>:</b>; relies only on the literal \n from <br>)
# ---------------------------------------------------------------------------
def extract_parties(cell):
    if cell is None:
        return ""
    raw_html = str(cell)
    raw_html = re.sub(r"<br\s*/?>", "\n", raw_html)
    fragment_soup = BeautifulSoup(raw_html, "html.parser")
    text = fragment_soup.get_text()
    lines = [normalize_text(line) for line in text.split("\n")]
    lines = [line for line in lines if line]
    return " | ".join(lines)


# ---------------------------------------------------------------------------
# 7. TOTAL CASES + PAGE COUNT DETECTION
# ---------------------------------------------------------------------------
def detect_total_cases(soup):
    text = soup.get_text(" ", strip=True)
    match = re.search(r"Total\s*:?\s*([\d.,]+)\s*Perkara", text, re.IGNORECASE)
    if not match:
        return None
    cleaned = match.group(1).replace(".", "").replace(",", "")
    try:
        return int(cleaned)
    except ValueError:
        return None


def estimate_pages(total_cases, per_page):
    if not total_cases or not per_page:
        return None
    return -(-total_cases // per_page)  # ceil division


# ---------------------------------------------------------------------------
# 8. ROW EXTRACTION
# ---------------------------------------------------------------------------
def extract_case_rows(soup):
    table = detect_case_table(soup)
    if table is None:
        return []

    mapping = get_header_mapping(table)
    idx_nomor = find_column_index(mapping, "Nomor Perkara")
    idx_tanggal = find_column_index(mapping, "Tanggal Register")
    idx_klasifikasi = find_column_index(mapping, "Klasifikasi Perkara")
    idx_pihak = find_column_index(mapping, "Para Pihak")
    idx_status = find_column_index(mapping, "Status Perkara")
    idx_lama = find_column_index(mapping, "Lama Proses")

    rows = table.find_all("tr")[1:]
    results = []
    for i, row in enumerate(rows, start=1):
        cells = row.find_all(["td", "th"])
        if not cells:
            continue

        def cell_text(idx):
            if idx is None or idx >= len(cells):
                return ""
            return normalize_text(cells[idx].get_text(" ", strip=True))

        link = ""
        anchor = row.find("a", href=True)
        if anchor:
            link = anchor["href"]

        para_pihak = extract_parties(cells[idx_pihak]) if idx_pihak is not None and idx_pihak < len(cells) else ""

        record = {
            "No": cell_text(0) or str(i),
            "Nomor Perkara": cell_text(idx_nomor),
            "Tanggal Register": cell_text(idx_tanggal),
            "Klasifikasi Perkara": cell_text(idx_klasifikasi),
            "Para Pihak": para_pihak,
            "Status Perkara": cell_text(idx_status),
            "Lama Proses": cell_text(idx_lama),
            "Link": link,
        }

        if record["Nomor Perkara"]:
            results.append(record)

    return results


# ---------------------------------------------------------------------------
# 9. SELENIUM DRIVER + PAGINATION (undetected_chromedriver + Cloudflare wait)
# ---------------------------------------------------------------------------
def build_driver(headless=False):
    options = uc.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options, version_main=CHROME_MAJOR_VERSION)
    driver.set_page_load_timeout(30)
    return driver


def _wait_for_cloudflare(driver, timeout=25):
    try:
        WebDriverWait(driver, timeout).until_not(EC.title_contains("Just a moment"))
    except TimeoutException:
        pass


def _find_next_page_element(driver, current_page):
    strategies = [
        (By.XPATH, f'//ul[contains(@class,"pagination")]//a[normalize-space(text())="{current_page + 1}"]'),
        (By.XPATH, '//a[contains(@class,"next") or @rel="next"]'),
        (By.XPATH, '//a[normalize-space(text())="»" or normalize-space(text())=">" '
                    'or normalize-space(text())="Next" or normalize-space(text())="Selanjutnya"]'),
        (By.XPATH, f'//a[normalize-space(text())="{current_page + 1}"]'),
    ]
    for by, selector in strategies:
        try:
            for el in driver.find_elements(by, selector):
                parent_classes = ""
                try:
                    parent_classes = el.find_element(By.XPATH, "./..").get_attribute("class") or ""
                except NoSuchElementException:
                    pass
                if "disabled" in parent_classes:
                    continue
                if el.is_displayed():
                    return el
        except Exception:
            continue
    return None


def _wait_for_table_refresh(driver, old_first_row_text, timeout=20):
    def _changed(drv):
        try:
            soup = BeautifulSoup(drv.page_source, "html.parser")
            table = detect_case_table(soup)
            if table is None:
                return False
            rows = table.find_all("tr")
            if len(rows) < 2:
                return False
            new_first_row_text = normalize_text(rows[1].get_text(" ", strip=True))
            return new_first_row_text != old_first_row_text and new_first_row_text != ""
        except StaleElementReferenceException:
            return False

    WebDriverWait(driver, timeout).until(_changed)


def scrape_all_pages(phi_url, headless=False):
    driver = build_driver(headless=headless)
    all_rows = []
    court_name = None
    total_expected = None

    try:
        driver.get(phi_url)
        _wait_for_cloudflare(driver)
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "table")))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        court_name = detect_court_name(soup)
        total_expected = detect_total_cases(soup)

        first_page_rows = extract_case_rows(soup)
        per_page = len(first_page_rows) or 20
        estimated_pages = estimate_pages(total_expected, per_page) or 1

        print(f"Pengadilan terdeteksi   : {court_name}")
        print(f"Total perkara terdeteksi: {total_expected}")
        print(f"Data per halaman        : {per_page}")
        print(f"Estimasi total halaman  : {estimated_pages}")

        page_num = 1
        print(f"Memproses halaman {page_num}/{estimated_pages} -> {len(first_page_rows)} data")
        all_rows.extend(first_page_rows)

        while total_expected is None or len(all_rows) < total_expected:
            table = detect_case_table(soup)
            rows = table.find_all("tr") if table else []
            old_first_row_text = normalize_text(rows[1].get_text(" ", strip=True)) if len(rows) > 1 else ""

            next_el = _find_next_page_element(driver, page_num)
            if next_el is None:
                print("Tidak ada tombol halaman berikutnya -- berhenti.")
                break

            try:
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_el)
                next_el.click()
            except Exception as e:
                print(f"Gagal klik halaman berikutnya: {e}")
                break

            _wait_for_cloudflare(driver)

            try:
                _wait_for_table_refresh(driver, old_first_row_text)
            except TimeoutException:
                print("Timeout menunggu halaman berikutnya termuat -- berhenti.")
                break

            time.sleep(POLITE_DELAY)
            page_num += 1
            soup = BeautifulSoup(driver.page_source, "html.parser")
            page_rows = extract_case_rows(soup)
            print(f"Memproses halaman {page_num}/{estimated_pages} -> {len(page_rows)} data")
            all_rows.extend(page_rows)

            if page_num > estimated_pages + 2:
                print("Melebihi estimasi jumlah halaman -- berhenti sebagai pengaman.")
                break

    finally:
        driver.quit()

    seen = set()
    deduped = []
    for row in all_rows:
        key = row.get("Nomor Perkara") or row.get("Link")
        if key in seen:
            continue
        seen.add(key)
        deduped.append(row)

    return deduped, court_name, total_expected


# ---------------------------------------------------------------------------
# 10. EXPORT
# ---------------------------------------------------------------------------
def export_results(rows, court_name):
    os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)
    os.makedirs(OUTPUT_EXCEL_DIR, exist_ok=True)

    safe_name = re.sub(r'[\\/*?:"<>|]', "", court_name).strip()
    filename = f"PHI_{safe_name}"

    df = pd.DataFrame(rows, columns=["No", "Nomor Perkara", "Tanggal Register",
                                      "Klasifikasi Perkara", "Para Pihak",
                                      "Status Perkara", "Lama Proses", "Link"])
    df["No"] = range(1, len(df) + 1)

    csv_path = os.path.join(OUTPUT_CSV_DIR, f"{filename}.csv")
    excel_path = os.path.join(OUTPUT_EXCEL_DIR, f"{filename}.xlsx")

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_excel(excel_path, index=False, engine="openpyxl")

    return csv_path, excel_path, df


# ---------------------------------------------------------------------------
# 11. RUN
# ---------------------------------------------------------------------------
def run_scraper(input_url: str, headless: bool = False):
    session = create_session()
    print("Mengakses situs untuk deteksi awal...")
    resp = session.get(input_url, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    phi_url = detect_phi_url(soup, input_url)
    print(f"Halaman PHI terdeteksi: {phi_url}")

    rows, court_name, total_expected = scrape_all_pages(phi_url, headless=headless)

    print("-" * 50)
    print(f"Total hasil scraper : {len(rows)}")
    if total_expected:
        print(f"Total diharapkan    : {total_expected}")
        if len(rows) != total_expected:
            print("PERINGATAN: jumlah hasil tidak sama dengan total yang diharapkan.")

    csv_path, excel_path, df = export_results(rows, court_name)
    print(f"CSV disimpan   : {csv_path}")
    print(f"Excel disimpan : {excel_path}")

    return rows, court_name, df

In [ ]:
rows, court_name, df = run_scraper(
    "https://sipp.pn-semarangkota.go.id/",
    headless=False,
)

df.head()